1. Importation des bibliothèques

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

2. Chargement des données

In [2]:
df = pd.read_csv("heart_disease_uci.csv")

print(df.head())
print(df.shape)

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  
(920, 16

3. Analyse exploratoire des données (EDA)

In [3]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    str    
 3   dataset   920 non-null    str    
 4   cp        920 non-null    str    
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    str    
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    str    
 13  ca        309 non-null    float64
 14  thal      434 non-null    str    
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 156.4+ KB


id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [4]:
num
df['target'] = (df['num'] > 0).astype(int)
plt.figure(figsize=(6,4))

sns.countplot(
    x='target',
    data=df
)

plt.title("Répartition des patients")
plt.show()

NameError: name 'num' is not defined

Corrélation des variables numériques

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    cmap='coolwarm'
)

plt.title("Matrice de corrélation")
plt.show()

4. Prétraitement des données
Suppression des colonnes inutiles

In [ ]:
df.drop(['id','num'], axis=1, inplace=True)

In [ ]:
# Séparation X et y
X = df.drop('target', axis=1)
y = df['target']

In [ ]:
# Colonnes numériques et catégorielles
num_features = X.select_dtypes(
    include=['int64','float64']
).columns

cat_features = X.select_dtypes(
    include=['object']
).columns

Pipeline de prétraitement
Variables numériques

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
# Variables catégorielles
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


In [ ]:
# Transformation globale
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

In [ ]:
# 5. Division Train/Test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# 6. Modèle de Régression Logistique

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])


In [ ]:
# Entraînement
model.fit(X_train, y_train)

In [ ]:
# 7. Prédictions
y_pred = model.predict(X_test)

In [ ]:
# 8. Évaluation du modèle
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

In [ ]:
# Precision
precision = precision_score(y_test, y_pred)

print("Precision :", precision)

In [ ]:
# Recall

recall = recall_score(y_test, y_pred)

print("Recall :", recall)

In [ ]:
# F1-score
f1 = f1_score(y_test, y_pred)

print("F1 Score :", f1)

In [ ]:
# Rapport de classification

print(classification_report(
    y_test,
    y_pred
))

In [ ]:
# 9. Matrice de confusion
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Prédiction")
plt.ylabel("Réel")
plt.title("Matrice de confusion")
plt.show()

10. Interprétation des résultats
Accuracy

Mesure le pourcentage global de prédictions correctes.

Precision

Indique la proportion des patients prédits malades qui sont réellement malades.

Recall

Mesure la capacité du modèle à détecter les patients atteints de maladie cardiaque.

F1-score

Compromis entre précision et rappel.

Conclusion (à mettre dans le rapport)
Rédaction

L'objectif de cette étude était de prédire la présence d'une maladie cardiaque à partir des caractéristiques médicales des patients du jeu de données Heart Disease UCI. Après une phase d'analyse exploratoire, les données ont été prétraitées en traitant les valeurs manquantes, en encodant les variables catégorielles et en normalisant les variables numériques.

Un modèle de régression logistique a ensuite été entraîné sur les données d'apprentissage puis évalué sur les données de test. Les performances ont été mesurées à l'aide de l'accuracy, de la précision, du rappel et du score F1. La matrice de confusion a permis de visualiser les bonnes et mauvaises classifications réalisées par le modèle.

Les résultats obtenus montrent que la régression logistique est capable d'identifier efficacement les patients présentant une maladie cardiaque. Ce modèle constitue donc une solution simple, interprétable et performante pour un problème de classification binaire dans le domaine médical.
